# 01 · Workflow — run any single node

<div style="background: rgba(128, 128, 128, 0.1); border-left: 4px solid rgba(128, 128, 128, 0.6); padding: 8px 14px; border-radius: 6px;">

This notebook is the day-to-day loop for **one node**, and the main way to run nodes in IQCC Learn:

1. download the **latest QUAM state and wiring** from IQCC,
2. pick **any node** from the library (default `05_T1`) — it runs on a random active qubit,
3. look at the plot and the fit — the **data is always saved**,
4. decide whether to **keep the state update** the node proposes.

To chain several nodes, a graph notebook will follow. If you have not yet seen what a node is made of, start with `00_start_here.ipynb`.

</div>

<div style="background: rgba(46, 160, 67, 0.1); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 Packages: **QUAlibrate** ([docs](https://qua-platform.github.io/qualibrate/)) runs the nodes; **QUAM** ([docs](https://qua-platform.github.io/quam/)) holds the state they read and update;
💎 **QUA** ([docs](https://docs.quantum-machines.co/latest/)) is the program each node runs on the OPX.

</div>

## 1 · Settings

In [ ]:
NODE = "05_T1"      # any name from the list printed in step 4
QUBITS = None       # None: one random active qubit of the machine; or e.g. ["q1", "q2"]
SIMULATE = False    # True: run on the OPX simulator instead of the hardware

## 2 · Backend and state location
Read from the active qualibrate config, the same source the nodes use, so it is the same in every checkout.

In [ ]:
from pathlib import Path

# Everything below comes from the active qualibrate config: the same source the nodes use at runtime.
from qualibrate.core.config.resolvers import get_quam_state_path
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path

_config = get_qualibrate_config(get_qualibrate_config_path())
BACKEND = _config.project
_configured = get_quam_state_path(_config)
if not _configured:
    raise RuntimeError("No QUAM state path configured. Run install.py (workstation) or learn_setup.py (Learn) first.")
# install.py / learn_setup.py store an absolute path; a relative one (hand-run `qualibrate config`)
# means "relative to the checkout root" (the kernel's working directory).
STATE_PATH = Path(_configured)
if not STATE_PATH.is_absolute():
    STATE_PATH = Path.cwd() / STATE_PATH
print(f"backend: {BACKEND}\nstate:   {STATE_PATH}")


## 3 · Download the latest state and wiring
Uses `iqcc_cloud_client` (IQCC credentials must already be configured). **This overwrites the local `state.json` and `wiring.json`.**

In [ ]:
import json
from iqcc_cloud_client import IQCC_Cloud

qc = IQCC_Cloud(BACKEND)
latest_wiring = qc.state.get_latest("wiring")
latest_state = qc.state.get_latest("state")

STATE_PATH.mkdir(parents=True, exist_ok=True)
with open(STATE_PATH / "wiring.json", "w") as f:
    json.dump(latest_wiring.data, f, indent=4)
with open(STATE_PATH / "state.json", "w") as f:
    json.dump(latest_state.data, f, indent=4)
print("Wrote wiring.json and state.json to", STATE_PATH)

## 4 · Open the calibration library

<div style="background: rgba(46, 160, 67, 0.1); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 **QUAlibrate** — [docs](https://qua-platform.github.io/qualibrate/) · `QualibrationLibrary` scans `calibrations/` and registers every node by name.

</div>

In [ ]:
import json
import random

import matplotlib.pyplot as plt
from IPython.display import display
from qualibrate import QualibrationLibrary

library = QualibrationLibrary.get_active_library()
print("available nodes:")
for name in sorted(library.nodes):
    print("  ", name)

# Pick the qubit(s): a random one of the machine's active qubits, unless QUBITS is set above.
# Read from state.json, so this works wherever the checkout runs (no `config` import needed).
_state = json.loads((STATE_PATH / "state.json").read_text())
_active = _state.get("active_qubit_names") or list(_state.get("qubits", {}))
if QUBITS is None:
    QUBITS = [random.choice(_active)]
print(f"\nnode:   {NODE}\nqubits: {QUBITS}   (active: {_active})")

## 5 · Run the node

<div style="background: rgba(56, 132, 255, 0.1); border-left: 4px solid rgba(56, 132, 255, 0.75); padding: 8px 14px; border-radius: 6px;">

💎 **QUA** — the node builds and runs a QUA program on the OPX (see `00_start_here.ipynb` §6 for how one is written).
🧩 **QUAlibrate** — `library.nodes[NODE].run(...)` executes all the node's run-actions, overriding the parameters you pass. The **data is saved** to `data/` as the last action.
🧩 **QUAM** — the node also writes its proposed update into the state. Below we show what changed, and §6 decides whether you keep it.

</div>

In [ ]:
# Keep a copy of the state on disk, so the update can be undone in step 6.
_backup = {f: (STATE_PATH / f).read_text() for f in ("state.json", "wiring.json")}

node = library.nodes[NODE]
summary = node.run(qubits=QUBITS, simulate=SIMULATE)   # returns a run summary; results live on `node`


def _flatten(d, prefix=""):
    for k, v in d.items():
        key = f"{prefix}.{k}" if prefix else str(k)
        if isinstance(v, dict):
            yield from _flatten(v, key)
        elif not isinstance(v, (list, tuple)):
            yield key, v


def show(node, summary):
    """Print outcomes, fit results and proposed state changes, then show the figures the node produced."""
    print("outcomes:", {q: getattr(o, "value", o) for q, o in summary.outcomes.items()})
    if summary.error:
        print("ERROR:", summary.error)
    for qubit, result in node.results.get("fit_results", {}).items():
        print(qubit, {k: v for k, v in result.items() if not hasattr(v, "__len__") or isinstance(v, str)})
    if node.snapshot_idx is not None:
        print("data saved as run id:", node.snapshot_idx)

    before = dict(_flatten({**json.loads(_backup["state.json"]), **json.loads(_backup["wiring.json"])}))
    after = dict(_flatten(node.machine.to_dict()))
    changes = {k: (before.get(k), v) for k, v in after.items() if before.get(k) != v}
    print(f"\nproposed state changes ({len(changes)}):")
    for key, (old, new) in changes.items():
        print(f"  {key}: {old} -> {new}")

    figures = dict(node.results.get("figures", {}))
    if "simulation" in node.results:
        figures["simulation"] = node.results["simulation"]["figure"]
    for name, fig in figures.items():
        print("figure:", name)
        display(fig)
    plt.close("all")


show(node, summary)

## 6 · Keep or discard the state update
The data of the run above is already saved. This cell only decides about the **QUAM state**:
`UPDATE_STATE = True` keeps the changes listed above, `False` puts `state.json` and `wiring.json` back as they were before the run.

In [ ]:
UPDATE_STATE = False   # True: keep the node's state update

if UPDATE_STATE:
    node.machine.save(STATE_PATH)
    print("state updated in", STATE_PATH)
else:
    for fname, text in _backup.items():
        (STATE_PATH / fname).write_text(text)
    print("state left unchanged (restored from the pre-run copy)")